# STEM Games 2026 — ragebait detector training on A100

This notebook trains the two classifiers and produces evaluation plots, then bundles everything for download back to your local PyCharm project.

**Expected total time on A100: 30–45 min** (vs 12 hours on CPU).

**Before you start**, zip your local project. From PowerShell in the project root:

```powershell
Compress-Archive -Path .\src,.\data,.\tests,.\compare_features.py,.\requirements.txt -DestinationPath project.zip -Force
```

This includes your code, the lexicons, the test fixtures, the corpus.jsonl you already built, and the ragebait_templates.jsonl. Excludes `.venv`, `models/`, `images/`, `.idea`, `__pycache__`. Should be a few MB.

Then run cells in order.

## 1. Verify you have an A100

If this shows T4 or anything else, go to `Runtime → Change runtime type → A100 GPU` and re-run.

In [ ]:
!nvidia-smi

## 2. Install dependencies

Colab already has CUDA-enabled torch pre-installed, so we skip torch here — reinstalling can sometimes downgrade to a CPU-only wheel and break GPU usage.

In [ ]:
!pip install -q transformers sentence-transformers scikit-learn "datasets<3.0" joblib tqdm python-dotenv nltk scipy pandas matplotlib sentencepiece protobuf

## 3. Upload your project zip

A file picker appears below — select `project.zip` from your local machine.

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import zipfile, os, shutil

zip_name = list(uploaded.keys())[0]
print(f"Unpacking {zip_name}...")
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('project')

# Find the project root (zip may or may not have a top-level folder)
root = 'project'
contents = os.listdir(root)
if len(contents) == 1 and os.path.isdir(os.path.join(root, contents[0])):
    root = os.path.join(root, contents[0])
os.chdir(root)
print(f"Working dir: {os.getcwd()}")
print(f"Top-level: {sorted(os.listdir('.'))}")
assert os.path.exists('src/train.py'), "Couldn't find src/train.py — check zip layout"
assert os.path.exists('data/corpus.jsonl'), "Couldn't find data/corpus.jsonl — forgot to include the corpus?"

## 4. Verify CUDA is reachable from Python

If this prints `CUDA: False`, something went wrong with torch — stop and check the install cell.

In [ ]:
import torch
print("torch version:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
    print("Memory:", torch.cuda.get_device_properties(0).total_memory / 1e9, "GB")

## 5. Quick corpus sanity check

Confirms the four label cells are all populated before we spend 30 min on training.

In [ ]:
from src.comment import load_corpus
import collections
c = collections.Counter((x.label_ai, x.label_ragebait) for x in load_corpus('data/corpus.jsonl'))
for (ai, rb), n in sorted(c.items()):
    print(f'  ai={ai}, rb={rb}: {n}')

## 6. Train both classifiers

This is the long block. On A100 expect:

- Perplexity baseline fit: ~30 sec
- Punctuation baseline fit: instant
- Topic-residual OLS fit: ~30 sec
- **Feature extraction across 1750 comments: 25–40 min** (DetectGPT dominates)
- LR fits + Platt calibration: ~30 sec

Total: ~30–45 min. The output is verbose — tqdm bar updates in place, so it should be readable.

In [ ]:
!python -u -m src.train --corpus data/corpus.jsonl

## 7. Smoke-test the trained classifier

Should produce non-NaN `p_ai`, `p_ragebait`, and `p_joint`, ideally with `p_ai > 0.6` and `p_ragebait > 0.6` on this AI-ragebait fixture.

In [ ]:
!python -m src.detect --input tests/comments/ragebait_ai_01.json

## 8. Run evaluation (ROC, PR, calibration, ablation)

Produces 8 PNGs and 2 CSVs in `images/`. Re-extracts features (it doesn't cache them — oversight in my code; not worth fixing today). Should take 25–40 min, similar to training.

In [ ]:
!python -u -m src.evaluate --corpus data/corpus.jsonl

## 9. Re-run compare_features now that baselines exist

This is a sanity-check we couldn't do before: with the trained punctuation baseline, `psi_7` should now be `OK` instead of `NaN`.

In [ ]:
!python compare_features.py

## 10. Bundle artifacts and download

Zips `models/` (trained classifier + baselines) and `images/` (8 PNGs + 2 CSVs) and triggers a download.

In [ ]:
import zipfile, os

out = 'trained_artifacts.zip'
with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as z:
    for d in ('models', 'images'):
        if not os.path.isdir(d):
            continue
        for root_, _, fs in os.walk(d):
            for f in fs:
                p = os.path.join(root_, f)
                z.write(p, p)
                print(' +', p)
print(f"\nWrote {out} ({os.path.getsize(out)/1e6:.1f} MB)")

from google.colab import files
files.download(out)

## Done

Back in your local project, unzip `trained_artifacts.zip` at the project root. You'll get:

- `models/classifiers.joblib` — the calibrated bundle
- `models/perplexity_baseline.json`, `models/punctuation_baseline.json`, `models/topic_residual_ols.json` — fitted baselines
- `images/roc_ai.png`, `roc_rb.png`, `pr_ai.png`, `pr_rb.png`, `calibration_ai.png`, `calibration_rb.png`, `ablation_ai.png`, `ablation_rb.png`
- `images/ablation_ai.csv`, `images/ablation_rb.csv`

From there:

```powershell
python -m src.detect --input tests/comments/ragebait_ai_01.json
```

should work locally without retraining — the loaded `.joblib` doesn't care that it was trained on GPU.